# Setup

In [ ]:
import pandas as pd
import string
import unicodedata
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

In [ ]:
LOTONE = chr(0x0300)
HITONE = chr(0x0301)
RISETONE = chr(0x030C)
MIDTONE1 = chr(0x0304)
MIDTONE2 = chr(0x0305)
TONECHARS = {LOTONE, HITONE, RISETONE,MIDTONE1,MIDTONE2}

UNDERDOT = chr(0x0323)
UNDERLINE = chr(0x0329)
UNDERDIACS = {UNDERDOT, UNDERLINE}

def remove_diacritics(input_str):
    '''
    Removes diacritical marks from a Unicode string.
    '''
    # normalize and fully decompose
    nfkd_form = unicodedata.normalize('NFD', input_str)

    # select all non-combining characters
    non_combining = [c for c in nfkd_form if not unicodedata.combining(c)]
    return ''.join(non_combining)

def normalize(input_str):
    '''
    Normalize into NFD form (fully decomposed)
    '''
    nfd = unicodedata.normalize('NFD', input_str)

    # convert underline to underdot
    return nfd.replace(chr(0x0329), chr(0x0323))

def eval_row_fixhalluc(row):
    '''
    Counts errors in a row
    Works even when a hallucination means everything is offset

    Output: pd.Series with all relevant diacritic+hallucination info
    '''
    pred = row['Predictions']
    target = row['Targets']
    source = row['Sources']
    pred_words = pred.split(' ')
    target_words = target.split(' ')
    source_words = source.split(' ')

    # if number of words doesn't match, return NaN
    # if len(pred_words) != len(target_words): return -1,-1,-1

    # calculate hallucinations and wrong diacritics
    total = 0
    hallucinations = 0
    wrong = 0
    LasL = 0
    LasM = 0
    LasH = 0
    MasL = 0
    MasM = 0
    MasH = 0
    HasL = 0
    HasM = 0
    HasH = 0
    missingDot = 0
    extraDot = 0
    otherMistake = 0
    dotAsDot = 0
    dotlessAsDotless = 0
    predi = 0
    targi = 0
    
    while (True):
        # at end of both
        # print(predi, len(pred_words), targi, len(target_words))
        if predi >= len(pred_words) and targi >= len(target_words):
            break
        # at end of prediction but not target, so add in hallucination count
        if predi >= len(pred_words):
            hallucinations += (len(target_words)-1) - targi
            wrong += (len(target_words)-1) - targi
            break
        # at end of target but not prediction, so add in hallucination count
        if targi >= len(target_words):
            hallucinations += (len(pred_words) - 1) - predi
            wrong += (len(pred_words) - 1) - predi
            break

        # not at end of either sentence yet, so proceed normally
        curr_pred_word = pred_words[predi]
        curr_target_word = target_words[targi]
        curr_source_word = source_words[targi]
        
        # look for hallucination
        no_diacs = remove_diacritics(curr_pred_word)
        if no_diacs != curr_source_word:
            hallucinations+=1
            wrong+=1
            total+=1

            # look for offset in either direction
            # did the LLM add a word?
            if (predi+1) < len(pred_words):
                next_pred_no_diacs = remove_diacritics(pred_words[predi+1])
                if next_pred_no_diacs == curr_source_word:
                    predi += 1
                    continue
            # did the LLM delete a word?
            if (targi+1) < len(target_words):
                next_targ_no_diacs = remove_diacritics(target_words[targi+1])
                if next_targ_no_diacs == no_diacs:
                    targi += 1
                    continue
            # otherwise just one word was hallucinated, so update both indices by 1
            predi += 1
            targi += 1
            continue
        
        # if word is punctuation, don't count it
        if len(curr_target_word) == 1 and curr_target_word in string.punctuation: 
            predi += 1
            targi += 1
            continue
        else: total+=1

        # look for wrong diacritics
        normal_pred = normalize(curr_pred_word)
        normal_true = normalize(curr_target_word)
        if normal_pred == normal_true:
            for i in range(len(normal_pred)):
                char = normal_pred[i]
                if char == HITONE: HasH+=1
                elif char == LOTONE: LasL +=1
                elif char == UNDERDOT: dotAsDot +=1
                elif char in ['s', 'e', 'o']:
                    if i < len(normal_pred) - 1:
                        if normal_pred[i+1] != UNDERDOT: dotlessAsDotless+=1
                    else:  dotlessAsDotless+=1
                if char in ['a', 'e', 'i', 'o', 'u']:
                    if i < len(normal_pred) - 2:
                        if normal_pred[i+1] not in TONECHARS and normal_pred[i+2] not in TONECHARS: MasM+=1
                    elif i < len(normal_pred) - 1:
                        if normal_pred[i+1] not in TONECHARS: MasM+=1
                    else:  MasM+=1
        if normal_pred != normal_true:
            wrong+=1
            # loop through word and find what is different
            i = 0 # pred index
            j = 0 # true index
            while i < len(normal_pred) and j < len(normal_true):
                pred_char = normal_pred[i]
                true_char = normal_true[j]
                # same char
                if pred_char == true_char:
                    if pred_char == HITONE: HasH+=1
                    elif pred_char == LOTONE: LasL +=1
                    elif pred_char == UNDERDOT: dotAsDot +=1
                    elif pred_char in ['s', 'e', 'o']:
                        if i < len(normal_pred) - 1:
                            if normal_pred[i+1] != UNDERDOT: dotlessAsDotless+=1
                        else:  dotlessAsDotless+=1
                    if pred_char in ['a', 'e', 'i', 'o', 'u']:
                        if i < len(normal_pred) - 2:
                            if normal_pred[i+1] not in TONECHARS and normal_pred[i+2] not in TONECHARS: MasM+=1
                        elif i < len(normal_pred) - 1:
                            if normal_pred[i+1] not in TONECHARS: MasM+=1
                        else:  MasM+=1
                    i+=1
                    j+=1
                # pred is a letter, true is a diacritic
                elif (unicodedata.combining(pred_char) == 0) and (unicodedata.combining(true_char) != 0):
                    if true_char == HITONE: HasM += 1
                    if true_char == LOTONE: LasM += 1
                    if true_char == UNDERDOT: missingDot += 1
                    j+=1
                # pred is diacritic, true is a letter
                elif (unicodedata.combining(pred_char) != 0) and (unicodedata.combining(true_char) == 0):
                    if pred_char == HITONE: MasH +=1
                    if pred_char == LOTONE: MasL +=1
                    if pred_char == UNDERDOT: extraDot += 1
                    i+=1
                # both are diacritics (more complex)
                # check for dots first
                elif pred_char == UNDERDOT:
                    extraDot+=1
                    i+=1
                elif true_char == UNDERDOT:
                    missingDot+=1
                    j+=1
                # now compare tones
                elif pred_char == HITONE and true_char == LOTONE:
                    LasH +=1
                    i+=1
                    j+=1
                elif pred_char == LOTONE and true_char == HITONE:
                    HasL += 1
                    i+=1
                    j+=1
                else:
                    print(pred_char, true_char)
                    print(curr_pred_word, curr_target_word)
                    otherMistake+=1
                    i+=1
                    j+=1
            if i < len(curr_pred_word):
                if curr_pred_word[i] == HITONE: MasH+=1
                elif curr_pred_word[i] == LOTONE: MasL+=1
                elif curr_pred_word[i] == UNDERDOT: extraDot +=1
                else: otherMistake+=1
            if j < len(curr_target_word):
                if curr_target_word[j] == HITONE: HasM+=1
                elif curr_target_word[j] == LOTONE: LasM +=1
                elif curr_target_word[j] == UNDERDOT: missingDot+=1
                else: otherMistake+=1
        predi += 1
        targi += 1
        
    return pd.Series({'Hallucinations' : hallucinations, 'Wrong Words' : wrong, 'Total Words' : total, 'L as L': LasL, 'L as M' : LasM, \
                        'L as H' : LasH, 'M as L' : MasL, 'M as M':MasM, 'M as H': MasH, 'H as L': HasL, 'H as M': HasM, 'H as H': HasH, \
                            'Missing Dot': missingDot, 'Extra Dot': extraDot, 'Dot as Dot': dotAsDot, 'Dotless as Dotless' : dotlessAsDotless,\
                                'Other Error': otherMistake})
   

# Run Evaluation

In [ ]:
# collect and display data
pred_df = pd.read_csv('test/pred.txt', header=None, names=['Predictions'])
targets_df = pd.read_csv('test/targets.txt', header=None, names=['Targets'])
sources_df = pd.read_csv('test/sources.txt', header=None, names=['Sources'])
test_df = pd.concat([sources_df, pred_df, targets_df], axis=1)
display(test_df)

In [ ]:
# calculate WER statistics
test_df = test_df.join(test_df.apply(lambda row: eval_row_fixhalluc(row), axis=1))
test_df['Hallucination WER'] = test_df.apply(lambda row: row['Hallucinations'] / row['Total Words'], axis=1)
test_df['WER'] = test_df.apply(lambda row: row['Wrong Words'] / row['Total Words'], axis=1)
display(test_df)

## Choose Random Examples for Analysis

In [ ]:
# Five highest hallucination rates
high_hallucinations = test_df.nlargest(5, 'Hallucination WER')
high_hallucinations.to_csv('top5_hallucinations.csv')

# Five highest WER
high_wer = test_df[test_df['Hallucinations'] == 0].nlargest(10, 'WER')
high_wer.to_csv('top5_wer.csv')

# Ten random examples
random = test_df.sample(10)
random.to_csv('llm_random.csv')

In [ ]:
display(high_hallucinations)
display(high_wer)
display(random)

# Analyze

In [ ]:
print(f"Hallucination WER (by sentence): {test_df['Hallucination WER'].mean()}")
print(f"Hallucination WER (by words): {test_df['Hallucinations'].sum() / test_df['Total Words'].sum()}")
print(f"Full WER (by sentence): {test_df['WER'].mean()}")
print(f"Full WER (by words): {test_df['Wrong Words'].sum() / test_df['Total Words'].sum()}")

In [ ]:
df = test_df

In [ ]:
# What percent words have WER/hallucination WER/etc below x?
test_val = .2

# can look at 'Hallucinations', 'Hallucination WER', 'Wrong Words', and 'WER'
subset = test_df[test_df['Hallucination WER'] < test_val]
print(f"Length of Sentence: {subset['Total Words'].mean()}")

wrong_count = len(subset)
total_count = len(test_df)
print(wrong_count, total_count)
print(wrong_count/total_count)

In [ ]:
# Graph tonal confusion matrix

# create matrix of raw counts
tone_results_raw = pd.DataFrame([
    [df['L as L'].sum(), df['L as M'].sum(), df['L as H'].sum()],
    [df['M as L'].sum(), df['M as M'].sum(), df['M as H'].sum()],
    [df['H as L'].sum(), df['H as M'].sum(), df['H as H'].sum()]
], columns=['Pred L', 'Pred M', 'Pred H'], index=['True L', 'True M', 'True H'])
display(tone_results_raw)

# create matrix of percentages (i.e. recall)
tone_results_pct = 100 * (tone_results_raw.div(tone_results_raw.sum(axis=1), axis=0))
display(tone_results_pct)

# Optional: Include percentages as annotation labels
annot_labels = pd.DataFrame(
    [[f"{int(tone_results_raw.iloc[i,j])}\n({tone_results_pct.iloc[i,j]:.2f}%)" 
      for j in range(3)] for i in range(3)],
    columns=tone_results_raw.columns,
    index=tone_results_raw.index
)

# Draw heatmap
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    tone_results_raw,          # color scale based on raw counts
    # annot=annot_labels,        # display percentages
    annot=tone_results_raw,
    annot_kws={'size': 14},
    fmt='',                    # needed when annot is strings
    cmap='Blues',
    linewidths=0.5,
    linecolor='#cccccc',
    ax=ax,
    cbar_kws={'label': 'Count'}
)

ax.set_title('Tone Confusion Matrix', fontsize=16, pad=10)
ax.set_xlabel('Predicted Diacritic', fontsize=14)
ax.set_ylabel('True Diacritic', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Graph Dot Confusion Matrix
dot_results_raw = pd.DataFrame([[df['Dot as Dot'].sum(), df['Missing Dot'].sum()], [df['Extra Dot'].sum(), df['Dotless as Dotless'].sum()]], 
                           columns=['Pred Dot', 'Pred Dotless'], index=['True Dot', 'True Dotless'])
display(dot_results_raw)

dot_results_pct = 100*dot_results_raw.div(dot_results_raw.sum(axis=1), axis=0)
display(dot_results_pct)

annot_labels = pd.DataFrame(
    [[f"{int(dot_results_raw.iloc[i,j])}\n({dot_results_pct.iloc[i,j]:.2f}%)" 
      for j in range(2)] for i in range(2)],
    columns=dot_results_raw.columns,
    index=dot_results_raw.index
)

# Draw heatmap — colored by raw counts, labeled with percentages
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    dot_results_raw,          # color scale based on raw counts
    # annot=annot_labels,
    annot_kws={"size": 14},
    fmt='',                    # needed when annot is strings
    cmap='Blues',
    linewidths=0.5,
    linecolor='#cccccc',
    ax=ax,
    cbar_kws={'label': 'Count'},
    
)

ax.set_title('Underdots Confusion Matrix', fontsize=16, pad=10)
ax.set_xlabel('Predicted Diacritic', fontsize=14)
ax.set_ylabel('True Diacritic', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Display distribution of WER
# can modify to show WER or Hallucination WER
# Group by 10% bins
bins = np.arange(0, 1.1, 0.1)
bin_labels = [f"{int(b*100)}-{int((b+0.1)*100)}%" for b in bins[:-1]]

# count WER
# counts, _ = np.histogram(df['WER'], bins=bins)
counts, _ = np.histogram(df['Hallucination WER'], bins=bins)

# graph
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(bin_labels, counts)

# Add count labels on top of bars
for bar, count in zip(bars, counts):
    if count > 0:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.05,
            str(count),
            ha='center', va='bottom'
        )

ax.set_title('Hallucination Rate Distribution by Sentences')
ax.set_xlabel('Hallucination WER (%)')
ax.set_ylabel('Number of Sentences')
plt.show()